### **Task 1: Conceptual Understanding**

**1. What is the difference between "Love" and "love" in NLP?**
In a computer's interpretation, "Love" (with an uppercase 'L') is a different token than "love" because they have different ASCII/Unicode representational values. If we don't convert the text to lowercase prior to processing, the model will view them as separate words and split their frequency counts. This decreases how well the model will understand the true meaning of the word.

**2. What happens if stopwords are not removed?**
Stopwords (for example: "the", "is", "and") are highly common but contribute insignificant meaning to the semantics of language. When these words aren't removed from text before analysis, they produce noise and create extra dimensions in your data. This uses unnecessary memory and makes machine learning training take longer without adding value to the predictive process.

**3. Provide two real-world scenarios where removing stopwords can be harmful.**
* **Sentiment Analysis:** Removing words like 'not' or 'very' can completely reverse the meaning of a sentence. For instance, the negative sentence *"The movie was not good"* turns into the positive phrase *"movie good"*.
* **Language Translation or Chatbots:** Sequence-to-sequence models use stopwords to generate grammatically correct, human-readable sentences that make sense in context. Removing stopwords destroys this structural context.

**4. Explain the difference between stemming and lemmatization.**
* **Stemming:** This process involves chopping off parts of words in a systematic, rule-driven way (e.g., removing letters from the end to arrive at a base, like turning 'running' into 'run'). However, it can produce stems that aren't actual words (e.g., getting 'car' from 'caring'). It is computationally fast but can produce nonsensical results.
* **Lemmatization:** This method uses dictionaries and vocabulary analysis to determine a word's true base form, or lemma (e.g., correctly mapping 'better' to 'good'). Lemmatization produces valid, highly accurate words, but it takes much longer to process than basic stemming.

# Tasks 2 to 7: Complete Python Implementation

In [1]:
import re
import string
from collections import Counter

# ==========================================
# Task 2: Build Advanced Preprocessing Function
# ==========================================
def preprocess_text(text):
    """
    Cleans raw text by removing URLs, numbers, punctuation, extra spaces,
    handling repeated characters, and filtering short tokens.
    """
    # Task 7: Error Handling (Handle empty or non-string inputs)
    if not isinstance(text, str) or not text.strip():
        return [], ""

    # 1. Remove URLs and email-like patterns
    text = re.sub(r'http\S+|www\.\S+', '', text)
    text = re.sub(r'\S+@\S+', '', text)

    # 2. Remove numbers
    text = re.sub(r'\d+', '', text)

    # 3. Convert text to lowercase
    text = text.lower()

    # 4. Remove Emojis and Punctuation (Keep only alphabets and spaces)
    # This also naturally handles the "Only emojis" error case by wiping them out
    text = re.sub(r'[^a-z\s]', ' ', text)

    # 5. Handle repeated characters (Reduce 3+ consecutive characters to 2)
    # Example: "looooved" -> "looved", "soooo" -> "soo"
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)

    # 6. Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()

    # 7. Tokenize and Remove very short tokens
    tokens = text.split()

    # Exceptions rule: kept 'is' as an exception to match the prompt's example:
    # "WOW!!! This is GREAT!!!" -> "wow this is great"
    exceptions = {"no", "not", "is", "ok", "me", "am"}
    filtered_tokens = [word for word in tokens if len(word) > 2 or word in exceptions]

    clean_sentence = " ".join(filtered_tokens)

    return filtered_tokens, clean_sentence


# ==========================================
# Task 6: Build Full Pipeline
# ==========================================
def full_pipeline(text_list):
    """
    Processes a list of raw text strings and returns a dictionary
    containing all tokens and cleaned sentences.
    """
    all_tokens = []
    all_clean_sentences = []

    for text in text_list:
        tokens, clean_sentence = preprocess_text(text)
        all_tokens.append(tokens)
        all_clean_sentences.append(clean_sentence)

    return {
        "tokens": all_tokens,
        "clean_sentences": all_clean_sentences
    }


# ==========================================
# Task 3 & 4: Stress Testing & Token Analytics
# ==========================================
sample_inputs = [
    "Get 100% FREE access now!!!",
    "I absolutely looooved this product 😍😍",
    "Worst service ever... 0/10",
    "Call me at 9876543210",
    "This is THE best course!!!",
    "Visit https://openai.com now!",
    "Nooooo this is baaad!!!",
    "OK OK OK I got it",
    "Win $$$ now!!! Limited offer!!!",
    "I am not happy with this",
    "",           # Task 7: Empty string test
    "😍😍😍",     # Task 7: Only emojis test
    "123 456"     # Task 7: Only numbers test
]

pipeline_results = full_pipeline(sample_inputs)
all_combined_tokens = []

print("--- TASK 3 & 4: STRESS TESTING & ANALYTICS ---\n")

for i, original in enumerate(sample_inputs):
    tokens = pipeline_results["tokens"][i]
    clean_sent = pipeline_results["clean_sentences"][i]
    all_combined_tokens.extend(tokens)

    # Analytics Calculations
    total_tokens = len(tokens)
    unique_tokens = len(set(tokens))
    avg_length = sum(len(word) for word in tokens) / total_tokens if total_tokens > 0 else 0

    print(f"Original Text : {original}")
    print(f"Cleaned Tokens: {tokens}")
    print(f"Cleaned Sent  : {clean_sent}")
    print(f"Analytics     : Total: {total_tokens} | Unique: {unique_tokens} | Avg Len: {avg_length:.2f}")
    print("-" * 50)


# ==========================================
# Task 5: Frequency Analysis
# ==========================================
print("\n--- TASK 5: FREQUENCY ANALYSIS ---")
word_counts = Counter(all_combined_tokens)

top_10 = word_counts.most_common(10)
# To get the least frequent, we slice the most_common list from the end
least_5 = word_counts.most_common()[:-6:-1]

print("\nTop 10 most frequent words:")
for word, count in top_10:
    print(f" - {word}: {count}")

print("\nTop 5 least frequent words:")
for word, count in least_5:
    print(f" - {word}: {count}")

--- TASK 3 & 4: STRESS TESTING & ANALYTICS ---

Original Text : Get 100% FREE access now!!!
Cleaned Tokens: ['get', 'free', 'access', 'now']
Cleaned Sent  : get free access now
Analytics     : Total: 4 | Unique: 4 | Avg Len: 4.00
--------------------------------------------------
Original Text : I absolutely looooved this product 😍😍
Cleaned Tokens: ['absolutely', 'looved', 'this', 'product']
Cleaned Sent  : absolutely looved this product
Analytics     : Total: 4 | Unique: 4 | Avg Len: 6.75
--------------------------------------------------
Original Text : Worst service ever... 0/10
Cleaned Tokens: ['worst', 'service', 'ever']
Cleaned Sent  : worst service ever
Analytics     : Total: 3 | Unique: 3 | Avg Len: 5.33
--------------------------------------------------
Original Text : Call me at 9876543210
Cleaned Tokens: ['call', 'me']
Cleaned Sent  : call me
Analytics     : Total: 2 | Unique: 2 | Avg Len: 3.00
--------------------------------------------------
Original Text : This is THE be

# Task 4: Token Analytics
## Analysis Questions

**Which sentence had the most noise?**

**Ans.**
The phrases `"Win $$$ now!!! Limited offer!!!"` and `"Worst service ever... 0/10"` contained the most noise when sampled. In both cases, they had a very small amount of readable words, and a large portion of the character count was non-alphabetical in nature (such as the `$$$`, `0/10`, and the extensive punctuation `...` and `!!!`).

A third example that also produced significant amounts of noise was `"Visit https://openai.com now!"`; in this instance, the majority of the string (the URL) was considered structural noise and removed entirely.

---

**Which sentence retained the most meaningful tokens after cleaning?**

**Ans.**
`"I absolutely looooved this product 😍😍"`

**Reasoning:**
While sentences like *"This is THE best course!!!"* and *"I am not happy with this"* retained the highest absolute count of tokens (5 each), they consist largely of smaller, grammatical words ('is', 'the', 'am', 'with').

On the other hand, *"I absolutely looooved this product"* retained 4 tokens (`['absolutely', 'looved', 'this', 'product']`), but these are highly descriptive, semantically rich words (adverbs, verbs, and nouns). This is backed up mathematically by the analytics script: it has the **highest average token length of 6.75**, proving that the tokens left over carry dense, meaningful information rather than just structural grammar.